# SDFusion: Single-view Reconstruction (img2shape)

In [1]:
# first set up which gpu to use
import os
gpu_ids = 0
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_ids}"

In [2]:
# import libraries
import numpy as np
from PIL import Image
from IPython.display import Image as ipy_image
from IPython.display import display
from termcolor import colored, cprint

import torch
import torch.backends.cudnn as cudnn
cudnn.benchmark = True
import torchvision.utils as vutils

from models.base_model import create_model
from utils.util_3d import render_sdf, render_mesh, sdf_to_mesh, save_mesh_as_gif, render_pcd

%load_ext autoreload
%autoreload 

In [3]:
# options for the model. please check `utils/demo_util.py` for more details
from utils.demo_util import SDFusionImage2ShapeOpt

seed = 2025
opt = SDFusionImage2ShapeOpt(gpu_ids=gpu_ids, seed=seed)
device = opt.device


[*] SDFusionImage2ShapeOption initialized.


In [4]:
# initialize SDFusion model
#ckpt_path = 'saved_ckpt/sdfusion-img2shape.pth'
#ckpt_path = '/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/Logs_GT/2025-10-12T20-33-56-sdfusion_model_img2shape-building-LR1e-5/ckpt/df_steps-latest.pth'
ckpt_path = '/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/Logs_GT/2025-10-28T22-19-52-sdfusion_model_img2shape-building-LR1e-5/ckpt/df_steps-latest.pth'
#ckpt_path ='/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/Logs_GT/2025-09-23T20-28-50-sdfusion_model_img2shape-building-LR1e-5/ckpt/df_steps-latest.pth' #this is wothout volume stacking only with controlent
opt.init_model_args(ckpt_path=ckpt_path)

SDFusion = create_model(opt)
cprint(f'[*] "{SDFusion.name()}" loaded.', 'cyan')

/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/external/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/models/sdfusion_model_img2shape.py:277: RuntimeWarning: divide by zero encountered in divide
  lvlb = betas**2 / (2 * self.posterior_variance.cpu().numpy() * alphas * (1 - alphas_cumprod))


Working with z of shape (1, 3, 16, 16, 16) = 12288 dimensions.
[*] VQVAE: weight successfully load from: /scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/logs_building/2025-05-19T19-58-28-vqvae-building-all-res64-LR1e-4-T0.2-release/ckpt/vqvae_steps-latest.pth
[*] weights loaded from /scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/Logs_GT/2025-10-28T22-19-52-sdfusion_model_img2shape-building-LR1e-5/ckpt/df_steps-latest.pth
[*] SDFusionImageFPShapeModel initialized (train=False).
[*] Model has been created: SDFusionImageFPShapeModel
[*] "SDFusionImageFPShapeModel" loaded.


## SDFusion: Single-view Reconstruction (img2shape)

TODO: add sample results here

In [ ]:
from utils.demo_util import preprocess_image
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch
import os
from PIL import Image
import numpy as np

# Output directory
out_dir = 'demo_results'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# Input image and mask paths
input_img = None
input_mask = "/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/data/BuildingNet_dataset_v0_1/footprints_png/train/COMMERCIALcastle_mesh2985.png"
#input_mask ="/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/data/BuildingNet_dataset_v0_1/footprints_png/test/COMMERCIALhotel_building_mesh0162.png"

# Preprocess image & mask
mean, std = [0.5] * 3, [0.5] * 3
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.Resize((256, 256)),
])

# This returns (masked_composite_img, clean_masked_img)
_, processed_img = preprocess_image(input_img, input_mask)
img_tensor = transform(processed_img).unsqueeze(0).to(device)  # Shape: [1, 3, 256, 256]

# Process mask to 2D binary tensor (channel-first)
if isinstance(input_mask, np.ndarray):
    mask_tensor = torch.from_numpy(input_mask).unsqueeze(0)
else:
    mask = Image.open(input_mask).convert('L')
    mask_tensor = transforms.ToTensor()(mask).unsqueeze(0)  # Shape: [1, 1, H, W]

# Resize to latent size (assumed 64x64) and binarize
D = 64  # You can extract this from vqvae config instead of hardcoding
mask_tensor = F.interpolate(mask_tensor, size=(D, D), mode='nearest')  # [1, 1, 64, 64]
mask_tensor = (mask_tensor > 0.5).float()

# Prepare dummy latent sdf input
zC = SDFusion.vqvae.ddconfig.z_channels
dummy_sdf = torch.zeros(1, zC, D, D, D).to(device)
# Ensure mask_tensor has 3 channels for CLIP compatibility
if mask_tensor.shape[1] == 1:
    mask_tensor = mask_tensor.repeat(1, 3, 1, 1)

fp = dummy_sdf[:, :1]  # if not already [B,1,D,H,W]
fp3d = torch.nn.functional.interpolate(fp, size=(8, 8, 8), mode='trilinear')
                
                
# Expand fp3d to match batch size of 2
batch_size = img_tensor.shape[0] * 2  # because unconditional + conditional = 2
fp3d = fp3d.expand(batch_size, -1, -1, -1, -1)  # Now: [2, 1, 8, 8, 8]


data = {
    'img': img_tensor,       # [1, 3, 256, 256]
    'fp': mask_tensor,       # [1, 3, 64, 64]
    'sdf': dummy_sdf,        # [1, 3, 64, 64, 64]
    'fp3d': fp3d             # now [1, 1, 8, 8, 8]
}
# For visual sanity check
display(processed_img)


In [ ]:
from utils.demo_util import preprocess_image
# img2shape
out_dir = 'demo_results'
if not os.path.exists(out_dir): os.makedirs(out_dir)

# input image. please use the grab_cut.ipynb to get the mask for your onw image
#input_img = "demo_data/revolving-chair.jpg"
input_img = None
#input_mask# = "demo_data/revolving-chair-mask.png"
input_mask = "/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/data/BuildingNet_dataset_v0_1/footprints_png/train/COMMERCIALcastle_mesh2985.png"

img, _ = preprocess_image(input_img, input_mask)

display(img)

In [ ]:
ngen = 1 # number of generated shapes
ddim_steps = 100
ddim_eta = 0.
uc_scale = 3.
print("img_tensor:", img_tensor.shape)
print("mask_tensor:", mask_tensor.shape)
print("dummy_sdf:", dummy_sdf.shape)
print("fp3d:", fp3d.shape)


from utils.util_3d import init_points_renderer

points_renderer = init_points_renderer(image_size=256, device=device)


#sdf_gen = SDFusion.img2shape(image=input_img, mask=input_mask, ddim_steps=ddim_steps, ddim_eta=ddim_eta, uc_scale=uc_scale)
sdf_gen = SDFusion.inference(
    data=data,
    ddim_steps=ddim_steps,
    ddim_eta=ddim_eta,
    uc_scale=uc_scale
)
mesh_gen = sdf_to_mesh(sdf_gen)

# 1. Static Mesh Render
mesh_img_tensor = render_mesh(SDFusion.renderer, mesh_gen)
mesh_img_np = mesh_img_tensor[0].permute(1, 2, 0).cpu().numpy()
mesh_img_np = (mesh_img_np * 255).astype(np.uint8)
Image.fromarray(mesh_img_np)

# # 2. Point Cloud Render
# pcd_img_tensor = render_pcd(points_renderer, verts)
# pcd_img_np = pcd_img_tensor[0].permute(1, 2, 0).cpu().numpy()
# pcd_img_np = (pcd_img_np * 255).astype(np.uint8)
# Image.fromarray(pcd_img_np)


# # 3. Optional: Matplotlib scatter plot
# import matplotlib.pyplot as plt
# fig = plt.figure(figsize=(6, 6))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(verts[0][:, 0].cpu(), verts[0][:, 1].cpu(), verts[0][:, 2].cpu(), s=1, c='blue')
# ax.set_title("Mesh Vertices as Point Cloud")
# plt.show()


# vis as gif
gen_name = f'{out_dir}/img2shape.gif'
save_mesh_as_gif(SDFusion.renderer, mesh_gen, nrow=3, out_name=gen_name)

#display(img)
display(ipy_image(gen_name))


# 1. Static Mesh Render
mesh_img = Image.fromarray(mesh_img_np)
display(mesh_img)

# 2. Point Cloud Render
pcd_img = Image.fromarray(pcd_img_np)
display(pcd_img)




In [ ]:
# Loop over all footprint masks, generate meshes, render top view, and compute metrics.
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw
import numpy as np
import pandas as pd
from tqdm import tqdm

# Assume SDFusion model is already loaded and in variable SDFusion, and device is defined.
# Also assume preprocess_image, sdf_to_mesh, and metric functions are available.

from utils.demo_util import preprocess_image  # for image/mask preprocessing
from utils.util_3d import sdf_to_mesh  # to convert predicted SDF to mesh (vertices, faces)
from utils.eval_metrics import compute_mask_metrics, clip_image_similarity

# Directories
input_dir = "/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/data/BuildingNet_dataset_v0_1/footprints_png/test"
output_dir = "./eval_results/controlnetonly"
os.makedirs(output_dir, exist_ok=True)

# Image preprocessing transforms
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize([0.5]*3, [0.5]*3)

# Generation parameters
ddim_steps = 100
ddim_eta = 0.0
uc_scale = 3.0

# Collect results
results = []

file_list = sorted([f for f in os.listdir(input_dir) if f.lower().endswith(".png")])
for file in tqdm(file_list, desc="Processing masks"):
    file_path = os.path.join(input_dir, file)
    
    # Preprocess footprint mask as model input
    _, processed_img = preprocess_image(None, file_path)
    # Resize to 256x256
    proc_img_resized = processed_img.resize((256, 256))
    # Convert to tensor and normalize to [-1,1]
    img_tensor = to_tensor(proc_img_resized).unsqueeze(0).to(device)
    img_tensor = normalize(img_tensor)
    
    # Load mask as grayscale, convert to tensor, resize to 64x64, and binarize
    mask_pil = Image.open(file_path).convert('L')
    mask_tensor = to_tensor(mask_pil).unsqueeze(0).to(device)  # [1,1,H,W]
    mask_tensor = F.interpolate(mask_tensor, size=(64, 64), mode='nearest')
    mask_tensor = (mask_tensor > 0.5).float()  # binary mask [1,1,64,64]
    # Repeat to 3 channels
    if mask_tensor.shape[1] == 1:
        mask_tensor = mask_tensor.repeat(1, 3, 1, 1)  # [1,3,64,64]
    
    # Prepare dummy SDF input
    zC = SDFusion.vqvae.ddconfig.z_channels
    dummy_sdf = torch.zeros((1, zC, 64, 64, 64), device=device)
    # Extract footprint 3D latent and expand for unconditional+conditional
    fp = dummy_sdf[:, :1]  # [1,1,64,64,64]
    fp3d = F.interpolate(fp, size=(8, 8, 8), mode='trilinear')  # [1,1,8,8,8]
    batch_size = img_tensor.shape[0] * 2  # unconditional + conditional
    fp3d = fp3d.expand(batch_size, -1, -1, -1, -1)  # [2,1,8,8,8]
    
    data = {
        'img': img_tensor,      # [1,3,256,256]
        'fp': mask_tensor,      # [1,3,64,64]
        'sdf': dummy_sdf,       # [1,zC,64,64,64]
        'fp3d': fp3d            # [2,1,8,8,8]
    }
    
    # Run the model inference
    sdf_gen = SDFusion.inference(data=data, ddim_steps=ddim_steps, ddim_eta=ddim_eta, uc_scale=uc_scale)
    # Convert SDF to mesh (returns vertices and faces)
    mesh_gen = sdf_to_mesh(sdf_gen)
 
    if mesh_gen is None:
        print(f"[Warning] sdf_to_mesh returned None for {file}, skipping.")
        continue
    # Extract vertices (N,3) and faces (M,3)
    # This may vary depending on implementation; assume it's a tuple (verts, faces)
    if isinstance(mesh_gen, (tuple, list)):
        verts, faces = mesh_gen
    else:
    # Try accessing verts/faces from mesh object
        try:
            verts = mesh_gen.verts_list()[0].cpu().numpy()
            faces = mesh_gen.faces_list()[0].cpu().numpy()
        except Exception as e:
            print(f"[Error] Failed to extract mesh for {file}: {e}")
            continue
    
    if verts is None or faces is None or len(verts) == 0 or len(faces) == 0:
    
        print(f"[Warning] Empty mesh for {file}, skipping.")
        continue
    
    if np.any(faces >= len(verts)) or np.any(faces < 0):    
        print(f"[Warning] Invalid face indices in {file}, skipping.")
    
        continue


    # Save mesh for visual inspection
    mesh_save_path = os.path.join(output_dir, file[:-4] + "_mesh.obj")

# Save as .obj file (simple ASCII format)
    with open(mesh_save_path, 'w') as f:
        for v in verts:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for face in faces:
            # OBJ format is 1-indexed
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")

    # Convert to numpy
    if torch.is_tensor(verts):
        verts = verts.cpu().numpy()[0]  # drop batch if present
    if torch.is_tensor(faces):
        faces = faces.cpu().numpy()[0].astype(np.int64)
    
    # Generate top-down mask from mesh
    # Get x,y coordinates of vertices
    xs = verts[:, 0]
    ys = verts[:, 1]
    # Compute bounds
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    width = x_max - x_min
    height = y_max - y_min
    max_range = max(width, height)
    if max_range == 0:
        scale = 1.0
        offset_x = 0.0
        offset_y = 0.0
    else:
        max_range = max(width, height)
        scale = 255.0 / max(max_range, 1e-5)  # prevent division by zero or huge scale

        offset_x = (max_range - width) / 2.0
        offset_y = (max_range - height) / 2.0

    # Create blank image for the mesh footprint
    mesh_mask_img = Image.new('L', (256, 256), 0)
    draw = ImageDraw.Draw(mesh_mask_img)
    print(f"\nFile: {file}")
    print(f"Verts min/max: X[{xs.min():.2f}, {xs.max():.2f}], Y[{ys.min():.2f}, {ys.max():.2f}]")
    print(f"Scale: {scale:.2f}, Offset X: {offset_x:.2f}, Offset Y: {offset_y:.2f}")
    # Draw each face as a filled polygon
    for face in faces:
        pts = []
        for vid in face:
            x = verts[vid, 0]
            y = verts[vid, 1]
            px = (x - x_min + offset_x) * scale
            py = (y - y_min + offset_y) * scale
            # Invert y-axis for image coordinate (origin at top-left)
            py = 255 - py
            pts.append((px, py))
   

        draw.polygon(pts, fill=255)
    mesh_mask_np = np.array(mesh_mask_img)
    
    # Resize original mask to 256x256 and binarize
    orig_mask_resized = mask_pil.resize((256, 256))
    orig_mask_np = (np.array(orig_mask_resized) > 128).astype(np.uint8)
    mesh_mask_bin = (mesh_mask_np > 128).astype(np.uint8)
    
    # Compute metrics
    metrics = compute_mask_metrics(orig_mask_np, mesh_mask_bin)
    print("Metrics returned:", metrics)
    print("Keys:", metrics.keys())



    # Compute CLIP similarity (convert masks to RGB images)
    orig_rgb = Image.fromarray((orig_mask_np * 255).astype(np.uint8)).convert('RGB')
    mesh_rgb = Image.fromarray((mesh_mask_bin * 255).astype(np.uint8)).convert('RGB')
    clip_score = clip_image_similarity(orig_rgb, mesh_rgb)
    
    # Save rendered mesh top-view image
    save_path = os.path.join(output_dir, file[:-4] + "_mesh_top.png")
    mesh_mask_img.save(save_path)
    
    # Append results
    results.append({
    'filename': file,
    'Chamfer': float(metrics.get("iou_score", 0.0)),
    'Hausdorff': float(metrics.get("hausdorff_distance", 999.0)),
    'CLIP_similarity': float(clip_score)
    })

# Save metrics to CSV
df = pd.DataFrame(results)
csv_path = os.path.join(output_dir, "metrics.csv")
df.to_csv(csv_path, index=False)
print(f"Saved metrics to {csv_path}, and rendered images to {output_dir}.")


In [ ]:
import pandas as pd

df = pd.read_csv("./eval_results/metrics.csv")
print(df.describe())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(df['Chamfer'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of IoU (Chamfer) Scores')
plt.xlabel('IoU Score')
plt.ylabel('Number of Footprints')
plt.grid(True)
plt.show()


plt.figure(figsize=(6, 4))
plt.hist(df['Hausdorff'], bins=20, color='salmon', edgecolor='black')
plt.title('Distribution of Hausdorff Distances')
plt.xlabel('Hausdorff Distance')
plt.ylabel('Number of Footprints')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df['CLIP_similarity'], bins=20, color='lightgreen', edgecolor='black')
plt.title('Distribution of CLIP Similarity Scores')
plt.xlabel('CLIP Similarity')
plt.ylabel('Number of Footprints')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['Chamfer'], df['Hausdorff'], alpha=0.7)
plt.title('IoU vs Hausdorff Distance')
plt.xlabel('IoU (Chamfer Score)')
plt.ylabel('Hausdorff Distance')
plt.grid(True)
plt.show()


In [5]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw
import numpy as np
import pandas as pd
from utils.demo_util import preprocess_image
from utils.util_3d import sdf_to_mesh
from utils.eval_metrics import compute_mask_metrics, clip_image_similarity


def process_single_mask(
    mask_path: str,
    output_dir: str,
    SDFusion,
    device,
    ddim_steps: int = 100,
    ddim_eta: float = 0.0,
    uc_scale: float = 3.0,
    save_mesh_obj: bool = True,
    verbose: bool = True
):
    """
    Process a single footprint mask and save results.
    """

    os.makedirs(output_dir, exist_ok=True)

    # Transforms
    to_tensor = transforms.ToTensor()
    normalize = transforms.Normalize([0.5]*3, [0.5]*3)

    # --- Preprocess ---
    filename = os.path.basename(mask_path)
    stem = os.path.splitext(filename)[0]

    _, processed_img = preprocess_image(None, mask_path)
    proc_img_resized = processed_img.resize((256, 256))
    img_tensor = to_tensor(proc_img_resized).unsqueeze(0).to(device)
    img_tensor = normalize(img_tensor)

    mask_pil = Image.open(mask_path).convert('L')
    mask_tensor = to_tensor(mask_pil).unsqueeze(0).to(device)
    mask_tensor = F.interpolate(mask_tensor, size=(64, 64), mode='nearest')
    mask_tensor = (mask_tensor > 0.5).float()
    mask_tensor = mask_tensor.repeat(1, 3, 1, 1)

    # Dummy SDF input
    zC = SDFusion.vqvae.ddconfig.z_channels
    dummy_sdf = torch.zeros((1, zC, 64, 64, 64), device=device)
    fp = dummy_sdf[:, :1]
    fp3d = F.interpolate(fp, size=(8, 8, 8), mode='trilinear')
    fp3d = fp3d.expand(2, -1, -1, -1, -1)

    data = {'img': img_tensor, 'fp': mask_tensor, 'sdf': dummy_sdf, 'fp3d': fp3d}

    # --- Inference ---
    with torch.no_grad():
        sdf_gen = SDFusion.inference(data=data, ddim_steps=ddim_steps, ddim_eta=ddim_eta, uc_scale=uc_scale)

    # --- Convert SDF to mesh ---
    mesh_gen = sdf_to_mesh(sdf_gen)
    if mesh_gen is None:
        print("[Warning] sdf_to_mesh returned None")
        return None

    try:
        if isinstance(mesh_gen, (tuple, list)):
            verts, faces = mesh_gen
        else:
            verts = mesh_gen.verts_list()[0].cpu().numpy()
            faces = mesh_gen.faces_list()[0].cpu().numpy()
    except Exception as e:
        print(f"[Error] Failed to extract mesh: {e}")
        return None

    if verts is None or len(verts) == 0:
        print("[Warning] Empty mesh, skipping.")
        return None

    if np.any(faces >= len(verts)) or np.any(faces < 0):
        print("[Warning] Invalid face indices.")
        return None

    # --- Save OBJ ---
    if save_mesh_obj:
        mesh_save_path = os.path.join(output_dir, f"{stem}_mesh.obj")
        with open(mesh_save_path, 'w') as f:
            for v in verts:
                f.write(f"v {v[0]} {v[1]} {v[2]}\n")
            for tri in faces:
                f.write(f"f {tri[0]+1} {tri[1]+1} {tri[2]+1}\n")

    # --- Top-view render ---
    xs, ys = verts[:, 0], verts[:, 1]
    x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
    width, height = x_max - x_min, y_max - y_min
    max_range = max(width, height)
    scale = 255.0 / max(max_range, 1e-5)
    offset_x = (max_range - width) / 2.0
    offset_y = (max_range - height) / 2.0

    mesh_mask_img = Image.new('L', (256, 256), 0)
    draw = ImageDraw.Draw(mesh_mask_img)
    for tri in faces:
        pts = []
        for vid in tri:
            x, y = verts[vid, 0], verts[vid, 1]
            px = (x - x_min + offset_x) * scale
            py = 255 - (y - y_min + offset_y) * scale
            pts.append((px, py))
        draw.polygon(pts, fill=255)

    mesh_mask_np = np.array(mesh_mask_img)

    # --- Original mask resized ---
    orig_mask_resized = mask_pil.resize((256, 256))
    orig_mask_np = (np.array(orig_mask_resized) > 128).astype(np.uint8)
    mesh_mask_bin = (mesh_mask_np > 128).astype(np.uint8)

    # --- Metrics ---
    metrics = compute_mask_metrics(orig_mask_np, mesh_mask_bin)
    orig_rgb = Image.fromarray((orig_mask_np * 255).astype(np.uint8)).convert('RGB')
    mesh_rgb = Image.fromarray((mesh_mask_bin * 255).astype(np.uint8)).convert('RGB')
    clip_score = clip_image_similarity(orig_rgb, mesh_rgb)

    top_view_path = os.path.join(output_dir, f"{stem}_mesh_top.png")
    mesh_mask_img.save(top_view_path)

    results = [{
        'filename': filename,
        'IoU': float(metrics.get("iou_score", 0.0)),
        'Hausdorff': float(metrics.get("hausdorff_distance", 999.0)),
        'CLIP_similarity': float(clip_score)
    }]
    df = pd.DataFrame(results)
    csv_path = os.path.join(output_dir, f"{stem}_metrics.csv")
    df.to_csv(csv_path, index=False)

    print(f"Saved mesh top view → {top_view_path}")
    print(f"Saved metrics → {csv_path}")
    return results[0]


In [15]:
metrics = process_single_mask(
    mask_path="/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/data/BuildingNet_dataset_v0_1/footprints_png/train/RESIDENTIALvilla_mesh4268.png",
    output_dir="/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/single_mask_outputs/volume_stacking",
    SDFusion=SDFusion,
    device=device
)
print(metrics)


Selected timesteps for ddim sampler: [  1  11  21  31  41  51  61  71  81  91 101 111 121 131 141 151 161 171
 181 191 201 211 221 231 241 251 261 271 281 291 301 311 321 331 341 351
 361 371 381 391 401 411 421 431 441 451 461 471 481 491 501 511 521 531
 541 551 561 571 581 591 601 611 621 631 641 651 661 671 681 691 701 711
 721 731 741 751 761 771 781 791 801 811 821 831 841 851 861 871 881 891
 901 911 921 931 941 951 961 971 981 991]
Selected alphas for ddim sampler: a_t: tensor([0.9983, 0.9895, 0.9804, 0.9708, 0.9609, 0.9505, 0.9398, 0.9287, 0.9171,
        0.9052, 0.8930, 0.8804, 0.8674, 0.8540, 0.8404, 0.8264, 0.8121, 0.7975,
        0.7827, 0.7675, 0.7521, 0.7365, 0.7207, 0.7047, 0.6885, 0.6722, 0.6557,
        0.6391, 0.6224, 0.6056, 0.5888, 0.5720, 0.5551, 0.5383, 0.5215, 0.5048,
        0.4882, 0.4716, 0.4552, 0.4390, 0.4229, 0.4070, 0.3913, 0.3758, 0.3605,
        0.3456, 0.3308, 0.3164, 0.3023, 0.2885, 0.2750, 0.2618, 0.2490, 0.2366,
        0.2245, 0.2128, 0.2014, 0.190

DDIM Sampler:   0%|          | 0/100 [00:00<?, ?it/s]/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/sdfusion/lib/python3.9/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/sdfusion/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
DDIM Sampler: 100%|██████████| 100/100 [00:03<00:00, 32.34it/s]


Saved mesh top view → /scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/single_mask_outputs/volume_stacking/RESIDENTIALvilla_mesh4268_mesh_top.png
Saved metrics → /scratch/gilbreth/dsimhadr/GenerativeTowns/SDFusion/single_mask_outputs/volume_stacking/RESIDENTIALvilla_mesh4268_metrics.csv
{'filename': 'RESIDENTIALvilla_mesh4268.png', 'IoU': 0.0216217041015625, 'Hausdorff': 16.0, 'CLIP_similarity': 0.86083984375}
